# Deepfake Detection Preprocessing: Face Extraction
This notebook is responsible for extracting face regions from raw video files. 
Processing only faces instead of full frames significantly reduces data size 
and improves the model's ability to learn facial manipulation artifacts.

In [ ]:
# Requirement: Ensure the dataset is available locally.
# This script assumes the dataset is already present in the local directory.

In [ ]:
# Configuration and average frame count calculation
import json # For handling metadata in JSON format
import glob # For file pattern matching
import numpy as np # For numerical analysis
import cv2  # OpenCV for video processing
import copy # For deep copying objects

# Locate all MP4 video files in the extracted directory
video_files = glob.glob('/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/Real videos/*.mp4')

frame_count = [] # List to store frame counts of all videos
# Iterate through videos to calculate statistics and filter out short clips
for video_file in video_files:
  cap = cv2.VideoCapture(video_file) # Open video file
  # Check total frames; if less than 150, remove from processing list to ensure consistency
  if(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) < 150): # Get frame count property
    video_files.remove(video_file) # Remove video from list if too short
    continue # Skip to next video
  frame_count.append(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))) # Store valid frame count

print("frames", frame_count) # Output list of frame counts
print("Total number of videos: ", len(frame_count)) # Output total video count
print('Average frame per video:', np.mean(frame_count)) # Output average frame count


In [ ]:
# Generator function to yield frames one by one from a video file
def frame_extract(path):
  vidObj = cv2.VideoCapture(path) 
  success = 1
  while success:
      # read() returns success status and the actual frame (image)
      success, image = vidObj.read()
      if success:
          yield image

# Ensure the destination directory exists
!mkdir -p '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/FF_REAL_Face_only_data'

import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
import os
import matplotlib.pyplot as plt
import face_recognition # Main library for face detection
from tqdm.autonotebook import tqdm # Progress bar for the loop

# Core function: Detects faces in video frames and saves a new video containing only the faces
def create_face_videos(path_list, out_dir):
  # Check how many videos are already processed to allow for resuming if interrupted
  already_present_count = glob.glob(out_dir + '*.mp4') # List existing files
  print("No of videos already present ", len(already_present_count)) # Log progress
  
  for path in tqdm(path_list): # Iterate over videos with progress bar
    # Construct output path for the new face-only video
    out_path = os.path.join(out_dir, path.split('/')[-1]) # Use same filename in output dir
    file_exists = glob.glob(out_path) # Check if output already exists
    if(len(file_exists) != 0): # If it exists
      print("File Already exists: ", out_path) # Skip it
      continue
    
    frames = [] # Temporary frame buffer
    # Initialize OpenCV VideoWriter to save the face-only frames
    # MJPG codec, 30 FPS, 112x112 resolution
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc('M','J','P','G'), 30, (112,112)) # Setup writer
    
    # Extract frames and find faces
    for idx, frame in enumerate(frame_extract(path)): # Loop through frames
      # Process up to 150 frames per video
      if(idx <= 150): # Limit frame count
        frames.append(frame) # Add frame to buffer
        # Process in batches of 4 for better GPU utilization with face_recognition
        if(len(frames) == 4): # Wait until batch is full
          # batch_face_locations returns a list of face bounding boxes for each image
          faces = face_recognition.batch_face_locations(frames) # Efficient batch detection
          for i, face in enumerate(faces): # Iterate through detected faces
            if(len(face) != 0): # If a face was found
              # Get coordinates: top, right, bottom, left
              top, right, bottom, left = face[0] # Take first detected face
            try:
              # Crop the face from the original frame and resize it to 112x112
              face_img = frames[i][top:bottom, left:right, :] # Extract ROI
              out.write(cv2.resize(face_img, (112,112))) # Resize and write to file
            except:
              # Skip frames where face detection might have failed or cropping is out of bounds
              pass
          frames = [] # Clear buffer for next batch
    
    # Cleanup and release the video writer for the current video
    try:
      del top, right, bottom, left # Clear coordinate variables
    except:
      pass
    out.release() # Close the video file


In [ ]:
# Start the face extraction process
create_face_videos(video_files, '/home/albaloshi/Desktop/Deepfake_detection_using_deep_learning-master/Dataset/FF_REAL_Face_only_data/')